# Phase 6 — Screening and Matching

Screens the candidate pool and matches one never-retracted control to each
treated author.

**Inputs:** `data/interim/phase05_candidates.csv`,
`data/interim/phase04_author_queue.csv`, `data/interim/phase04_papers.csv`,
`data/raw/world_bank_income.csv`, the OpenAlex snapshot

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase06_candidates_screened.csv` | candidates with covariates and screening flags |
| `data/interim/phase06_matches.csv` | treated-control pairs with match distance |
| `data/interim/phase07_control_queue.csv` | matched controls, the Phase 7 input |

## The matching design

**Exact** on pseudo-retraction year, career band and country income group.
**Nearest-neighbour** on pre-retraction publication count, within calipers.

Exact matching on discrete covariates and nearest-neighbour on continuous ones
is more transparent than a propensity score: a reader can see precisely what a
control shares with its treated author.

Calipers bound the nearest-neighbour step. Without them the matcher pairs a
three-paper author with a sixty-paper one wherever the pool is thin, and
reports a match.

**One control per treated author, without replacement.** Reuse would give
closer matches but requires weighting in estimation.

## Career length is measured differently for the two groups

Treated authors have complete publication histories from Phase 4, so career
start is the earliest year observed. Candidates are screened from the `authors`
entity, whose `counts_by_year` covers a rolling window and understates career
length. Career band is coarse enough to tolerate the asymmetry; the underlying
year counts are not compared across groups.

In [1]:
import gc
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from snapshot import Snapshot, concat, strip_id

CANDIDATES = "data/interim/phase05_candidates.csv"
TREATED_QUEUE = "data/interim/phase04_author_queue.csv"
TREATED_PAPERS = "data/interim/phase04_papers.csv"
WORLD_BANK = "data/raw/world_bank_income.csv"

OUT_SCREENED = "data/interim/phase06_candidates_screened.csv"
OUT_MATCHES = "data/interim/phase06_matches.csv"
OUT_QUEUE = "data/interim/phase07_control_queue.csv"

MIN_TOTAL_WORKS = 3

MATCH_RATIO = 1
EXACT_ON = ["pseudo_retraction_year", "career_band", "income_group"]
CALIPER_PRE_PUBS = 0.5          # |log(a+1) - log(b+1)|
CALIPER_CAREER = 3              # years
WITH_REPLACEMENT = False
RANDOM_SEED = 42

PRE_WINDOW = 7

# Matching on a pre-retraction total alone pairs authors with the same volume
# but different trajectories: an author whose output is rising and one whose
# output is flat can share a seven-year count. The event study then attributes
# the difference in slope to the retraction.
#
# The window is therefore split. RECENT_WINDOW covers the years immediately
# before the event and EARLY_WINDOW the years before that; matching on both
# constrains level and direction together.
RECENT_WINDOW = 3               # years -3..-1
EARLY_WINDOW = 4                # years -7..-4
CALIPER_RECENT = 0.5            # |log(a+1) - log(b+1)|
CALIPER_EARLY = 0.5

# Growth is the log ratio of the two windows. Matching on it directly, rather
# than relying on the two levels alone, bounds how far the trajectories may
# differ.
CALIPER_GROWTH = 0.4

# An author with several retractions has a post-period containing more than one
# event. Phase 8 excludes them from the primary panel, so matching them would
# extract controls that are never used.
PRIMARY_ONLY = True

TIER_TO_GROUP = {"H": "higher", "UM": "higher", "LM": "lower", "L": "lower"}

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

pd.set_option("display.width", 220)
os.makedirs("data/interim", exist_ok=True)

print(f"match ratio       1:{MATCH_RATIO}")
print(f"exact on          {EXACT_ON}")
print(f"calipers          pre-pubs {CALIPER_PRE_PUBS} (log), "
      f"career {CALIPER_CAREER}y")
print(f"with replacement  {WITH_REPLACEMENT}")
print(f"seed              {RANDOM_SEED}")

match ratio       1:1
exact on          ['pseudo_retraction_year', 'career_band', 'income_group']
calipers          pre-pubs 0.5 (log), career 3y
with replacement  False
seed              42


## Income group

The World Bank publishes its historical classification wide: one row per
country, one column per fiscal year, with the header row holding calendar
years. Column labels are reconstructed positionally.

The four tiers are collapsed to two. Retaining four leaves too few low-income
authors to support exact matching, and exact matching on a category with a
handful of members discards most of them.

`pycountry` converts ISO alpha-2 codes, as returned by OpenAlex, to the alpha-3
codes the classification uses.

In [2]:
import csv as _csv

_FALLBACK_ISO2_TO_ISO3 = {
    "CN": "CHN", "US": "USA", "IN": "IND", "JP": "JPN", "GB": "GBR",
    "DE": "DEU", "FR": "FRA", "IT": "ITA", "ES": "ESP", "KR": "KOR",
    "BR": "BRA", "CA": "CAN", "AU": "AUS", "RU": "RUS", "IR": "IRN",
    "TR": "TUR", "PL": "POL", "NL": "NLD", "CH": "CHE", "SE": "SWE",
    "BE": "BEL", "AT": "AUT", "DK": "DNK", "NO": "NOR", "FI": "FIN",
    "PT": "PRT", "GR": "GRC", "CZ": "CZE", "IL": "ISR", "SG": "SGP",
    "MY": "MYS", "TH": "THA", "ID": "IDN", "VN": "VNM", "PK": "PAK",
    "BD": "BGD", "EG": "EGY", "SA": "SAU", "AE": "ARE", "ZA": "ZAF",
    "NG": "NGA", "KE": "KEN", "MX": "MEX", "AR": "ARG", "CL": "CHL",
    "CO": "COL", "PE": "PER", "TW": "TWN", "HK": "HKG", "NZ": "NZL",
    "IE": "IRL", "RO": "ROU", "HU": "HUN", "UA": "UKR", "RS": "SRB",
    "HR": "HRV", "SK": "SVK", "SI": "SVN", "BG": "BGR", "LT": "LTU",
    "LV": "LVA", "EE": "EST", "IQ": "IRQ", "JO": "JOR", "LB": "LBN",
    "QA": "QAT", "KW": "KWT", "OM": "OMN", "MA": "MAR", "TN": "TUN",
    "DZ": "DZA", "ET": "ETH", "GH": "GHA", "TZ": "TZA", "UG": "UGA",
    "PH": "PHL", "LK": "LKA", "NP": "NPL", "KZ": "KAZ", "UZ": "UZB",
    "CU": "CUB", "VE": "VEN", "EC": "ECU", "UY": "URY", "CR": "CRI",
    "MO": "MAC", "LU": "LUX", "IS": "ISL", "CY": "CYP", "MT": "MLT",
}


def _iso3_converter():
    try:
        import pycountry

        def conv(a2):
            try:
                return pycountry.countries.get(alpha_2=str(a2).upper()).alpha_3
            except Exception:
                return None
        return conv, "pycountry"
    except ImportError:
        return (lambda a2: _FALLBACK_ISO2_TO_ISO3.get(str(a2).upper()),
                "built-in table")


def build_income_lookup():
    """iso3 -> most recent income tier."""
    if not os.path.isfile(WORLD_BANK):
        print(f"  [!] {WORLD_BANK} not found; income_group unavailable")
        return {}

    rows = list(_csv.reader(open(WORLD_BANK, encoding="utf-8-sig")))
    fys, n = [], 89
    for _ in range(2, len(rows[0])):
        fys.append(f"FY{n % 100:02d}")
        n += 1
    years = [(1900 + int(f[2:]) if int(f[2:]) >= 80 else 2000 + int(f[2:])) - 1
             for f in fys]

    latest = {}
    for r in rows[2:]:
        if len(r) < 3 or not r[0].strip():
            continue
        iso3 = r[0].strip()
        for j, y in enumerate(years):
            v = r[2 + j].strip() if 2 + j < len(r) else ""
            if v in ("L", "LM", "UM", "H"):
                if iso3 not in latest or y > latest[iso3][0]:
                    latest[iso3] = (y, v)

    print(f"  income tiers for {len(latest):,} countries, "
          f"{min(years)}-{max(years)}")
    return {k: v[1] for k, v in latest.items()}


def assign_income(df, country_cols=("country", "last_country"), label="frame"):
    inc = build_income_lookup()
    if not inc:
        df["income"] = df["income_group"] = np.nan
        return df

    to_iso3, source = _iso3_converter()

    src = None
    for c in country_cols:
        if c in df.columns:
            src = df[c] if src is None else src.fillna(df[c])
    if src is None:
        print(f"  [!] no country column on {label}")
        df["income"] = df["income_group"] = np.nan
        return df

    codes = src.dropna().astype(str).str.upper().unique()
    iso_map = {c: to_iso3(c) for c in codes}
    df["income"] = src.astype(str).str.upper().map(iso_map).map(inc)
    df["income_group"] = df.income.map(TIER_TO_GROUP)

    print(f"  {label}: country present {src.notna().mean():.1%}, "
          f"income group assigned {df.income_group.notna().mean():.1%} "
          f"(codes via {source})")
    unmapped = [c for c, v in iso_map.items() if v is None]
    if unmapped:
        print(f"    {len(unmapped)} country codes did not resolve: "
              f"{unmapped[:8]}")
    return df


def career_band(years):
    return pd.cut(years, bins=[0, 5, 10, 20, np.inf],
                  labels=["early (<5y)", "mid (5-10y)",
                          "senior (10-20y)", "veteran (20y+)"],
                  right=False)

## Treated covariates

In [3]:
treated = pd.read_csv(TREATED_QUEUE, low_memory=False)
treated["author_id"] = treated.author_id.astype(str)
print(f"treated authors  {len(treated):,}")

if PRIMARY_ONLY and "n_retractions" in treated.columns:
    n0 = len(treated)
    treated = treated[treated.n_retractions == 1]
    print(f"  primary sample only: {n0:,} -> {len(treated):,} "
          f"({len(treated) / n0:.1%})")

papers = pd.read_csv(TREATED_PAPERS, usecols=["author_id", "work_id",
                                              "pub_year"], low_memory=False)
papers = papers.drop_duplicates(subset=["author_id", "work_id"])
papers["author_id"] = papers.author_id.astype(str)
papers = papers.dropna(subset=["pub_year"])
papers["pub_year"] = papers.pub_year.astype(int)

ry = treated.set_index("author_id").first_retraction_year
papers["retraction_year"] = papers.author_id.map(ry)
papers = papers.dropna(subset=["retraction_year"])

pre = papers[(papers.pub_year >= papers.retraction_year - PRE_WINDOW) &
             (papers.pub_year < papers.retraction_year)]
treated["pre_publications"] = (treated.author_id
                                 .map(pre.groupby("author_id").size())
                                 .fillna(0).astype(int))

recent = papers[(papers.pub_year >= papers.retraction_year - RECENT_WINDOW) &
                (papers.pub_year < papers.retraction_year)]
early = papers[(papers.pub_year >= papers.retraction_year - PRE_WINDOW) &
               (papers.pub_year < papers.retraction_year - RECENT_WINDOW)]
treated["pre_recent"] = (treated.author_id
                           .map(recent.groupby("author_id").size())
                           .fillna(0).astype(int))
treated["pre_early"] = (treated.author_id
                          .map(early.groupby("author_id").size())
                          .fillna(0).astype(int))
# Annualised, so the two windows are comparable despite differing lengths.
treated["pre_growth"] = (np.log1p(treated.pre_recent / RECENT_WINDOW)
                         - np.log1p(treated.pre_early / EARLY_WINDOW))

first_pub = papers.groupby("author_id").pub_year.min()
treated["first_pub_year_extract"] = treated.author_id.map(first_pub)
treated["career_years_at_retraction"] = (treated.first_retraction_year
                                         - treated.first_pub_year_extract)
treated = treated[treated.career_years_at_retraction >= 0]
treated["career_band"] = career_band(treated.career_years_at_retraction)

treated = assign_income(treated, country_cols=("country", "last_country"),
                        label="treated")

if "first_category" in treated.columns and "arm" not in treated.columns:
    treated["arm"] = treated.first_category

print(f"\ncareer band")
print(treated.career_band.value_counts().to_string())
print(f"\npre-retraction publications over {PRE_WINDOW} years")
print(treated.pre_publications.describe().round(1).to_string())
print(f"\nannualised rate, recent {RECENT_WINDOW}y and earlier {EARLY_WINDOW}y")
print(f"  recent  mean {treated.pre_recent.mean() / RECENT_WINDOW:.2f}/yr")
print(f"  early   mean {treated.pre_early.mean() / EARLY_WINDOW:.2f}/yr")
print(f"\npre-period growth, log ratio of the two rates")
print(treated.pre_growth.describe().round(3).to_string())
rising = float((treated.pre_growth > 0).mean())
print(f"  rising into the event: {rising:.1%} of treated authors")

treated authors  55,821
  primary sample only: 55,821 -> 48,799 (87.4%)
  income tiers for 218 countries, 1988-2026
  treated: country present 97.8%, income group assigned 97.8% (codes via pycountry)
    1 country codes did not resolve: ['XK']

career band
career_band
veteran (20y+)     22026
senior (10-20y)    16546
mid (5-10y)         7526
early (<5y)         2701

pre-retraction publications over 7 years
count    48799.0
mean        40.9
std         69.9
min          0.0
25%          8.0
50%         20.0
75%         47.0
max       3618.0

annualised rate, recent 3y and earlier 4y
  recent  mean 6.88/yr
  early   mean 5.06/yr

pre-period growth, log ratio of the two rates
count    48799.000
mean         0.285
std          0.576
min         -4.155
25%         -0.069
50%          0.288
75%          0.624
max          4.650
  rising into the event: 68.9% of treated authors


## Screening the candidate pool

Candidates are collapsed to one row each, then their publication counts and
career spans are read from the `authors` entity. The same works threshold is
applied to both groups.

Most candidates appear on a single paper in the pool. That is a consequence of
retaining every paper in each journal-year rather than a fixed number: the pool
reaches deep into occasional authorship, and the works threshold removes those
who cannot support an estimate.

In [4]:
cand = pd.read_csv(CANDIDATES, low_memory=False)
n_raw = len(cand)
cand = cand.drop_duplicates(subset=["author_id", "work_id"])
if len(cand) < n_raw:
    print(f"removed {n_raw - len(cand):,} duplicate author-work rows")

cand_authors = (cand.groupby("author_id")
                  .agg(n_candidate_papers=("work_id", "nunique"),
                       source_id=("source_id", "first"),
                       cand_pub_year=("pub_year", "first"),
                       position=("position", "first"),
                       is_corresponding=("is_corresponding", "first"),
                       country=("country", "first"))
                  .reset_index())
cand_authors["author_id"] = cand_authors.author_id.astype(str)
del cand
gc.collect()
print(f"unique candidates {len(cand_authors):,}")

removed 67,773 duplicate author-work rows
unique candidates 10,335,622


In [5]:
snap = Snapshot()
AUTHOR_COLS = ["id", "works_count", "cited_by_count", "counts_by_year",
               "last_known_institutions"]

# Only bare identifiers. The authors entity prefixes every id, so stripping on
# read halves the hash table the membership test builds; with ten million
# candidates that is the difference between a fast scan and a memory-bound one.
want = pa.array(sorted(set(cand_authors.author_id)), type=pa.string())

PREFIX_LEN = len("https://openalex.org/")


class Screener:
    """Applies the works threshold during the scan.

    Candidates failing it cannot support an estimate and are the majority of
    the pool, so retaining them through the scan would hold tens of millions of
    nested citation histories in memory for no purpose.
    """

    def __init__(self, min_works):
        self.min_works = min_works
        self.parts = []
        self.n_found = 0

    def handle(self, tbl, path):
        col = tbl.column("id")
        if isinstance(col, pa.ChunkedArray):
            col = col.combine_chunks()
        bare = pc.utf8_slice_codeunits(col, PREFIX_LEN)

        hit = pc.fill_null(pc.is_in(bare, value_set=want), False)
        rows = pc.indices_nonzero(hit)
        if not len(rows):
            return None
        self.n_found += len(rows)

        w = tbl.take(rows)
        keep = pc.indices_nonzero(pc.fill_null(
            pc.greater_equal(w.column("works_count"), self.min_works), False))
        if not len(keep):
            return None
        w = w.take(keep)

        cby = w.column("counts_by_year").to_pylist()
        insts = w.column("last_known_institutions").to_pylist()

        self.parts.append(pd.DataFrame({
            "author_id": pc.utf8_slice_codeunits(
                w.column("id").combine_chunks()
                if isinstance(w.column("id"), pa.ChunkedArray)
                else w.column("id"), PREFIX_LEN).to_pylist(),
            "works_count": w.column("works_count").to_pylist(),
            "cited_by_count": w.column("cited_by_count").to_pylist(),
            "first_pub_year": [min((c["year"] for c in r
                                    if c.get("year") is not None),
                                   default=np.nan) for r in cby],
            "last_pub_year": [max((c["year"] for c in r
                                   if c.get("year") is not None),
                                  default=np.nan) for r in cby],
            "last_country": [(x[0].get("country_code") if x else None)
                             for x in insts],
            # Compact rather than a dictionary per author: ten million dicts
            # of year-count pairs do not fit in memory, and the values are
            # needed only once per retraction year during the expansion.
            "counts_by_year": ["|".join(
                f"{c['year']}:{c.get('works_count', 0)}" for c in
                sorted((x for x in r if x.get("year") is not None),
                       key=lambda x: -x["year"])) for r in cby],
        }))
        return None


scr = Screener(MIN_TOTAL_WORKS)
t0 = time.time()
snap.scan("authors", AUTHOR_COLS, scr.handle, progress_every=400)
fetched = (pd.concat(scr.parts, ignore_index=True).drop_duplicates("author_id")
           if scr.parts else pd.DataFrame())
scr.parts.clear()

print(f"\nelapsed {(time.time() - t0) / 60:.1f} min")
print(f"candidates found in the snapshot  {scr.n_found:,} of "
      f"{len(cand_authors):,}")
print(f"passing at least {MIN_TOTAL_WORKS} works        {len(fetched):,} "
      f"({len(fetched) / max(scr.n_found, 1):.1%} of those found)")

scanning authors: 1,961 files, 49.1 GiB on disk
  projecting 5 of 20 columns
  400/1,961 files | 10.5M rows | kept 0 | 7 MiB/s | eta 121m
  800/1,961 files | 11.0M rows | kept 0 | 4 MiB/s | eta 217m
  1,200/1,961 files | 11.5M rows | kept 0 | 3 MiB/s | eta 296m
  1,600/1,961 files | 65.8M rows | kept 0 | 9 MiB/s | eta 59m
  done: 119,129,660 rows scanned, 0 kept, 48.8 min

elapsed 49.9 min
candidates found in the snapshot  10,333,941 of 10,335,622
passing at least 3 works        8,118,192 (78.6% of those found)


In [6]:
if fetched.empty:
    raise SystemExit("no candidate records passed screening")

sc = cand_authors.merge(fetched, on="author_id", how="inner")
del fetched, cand_authors
gc.collect()
sc["career_span"] = sc.last_pub_year - sc.first_pub_year
sc["passes"] = True

sc = assign_income(sc, country_cols=("last_country", "country"),
                   label="candidates")

print(f"\nusable pool  {len(sc):,}")
print(f"\nworks per candidate")
print(sc.works_count.describe().round(1).to_string())
print(f"\nincome group")
print(sc.income_group.value_counts(dropna=False).to_string())

  income tiers for 218 countries, 1988-2026
  candidates: country present 92.9%, income group assigned 92.9% (codes via pycountry)
    1 country codes did not resolve: ['XK']

usable pool  8,118,192

works per candidate
count    8118192.0
mean          51.2
std         1219.6
min            3.0
25%            8.0
50%           21.0
75%           53.0
max      1898731.0

income group
income_group
higher    6971438
NaN        576232
lower      570522


## Pre-retraction windows

A candidate's pre-retraction counts depend on which year they are compared
against, so the windows are recomputed per retraction year from a single
publication matrix.

In [7]:
YEAR_MIN, YEAR_MAX = 1990, 2026
YEARS = np.arange(YEAR_MIN, YEAR_MAX + 1)
YCOL = {int(y): i for i, y in enumerate(YEARS)}

# Publication counts as one int32 row per candidate and one column per year.
# The per-year windows below are then array slices rather than a pass over
# several million dictionaries, and the matrix is the only copy held for the
# duration of the matching.
PUBS = np.zeros((len(sc), len(YEARS)), dtype=np.int32)
for r, s in enumerate(sc.counts_by_year):
    if not isinstance(s, str) or not s:
        continue
    for part in s.split("|"):
        if ":" not in part:
            continue
        ys, _, ns = part.partition(":")
        try:
            j = YCOL.get(int(ys))
            if j is not None:
                PUBS[r, j] = int(ns)
        except ValueError:
            continue

print(f"publication matrix: {PUBS.shape[0]:,} candidates x "
      f"{PUBS.shape[1]} years ({PUBS.nbytes / 2**30:.2f} GiB)")

sc = sc.drop(columns=["counts_by_year"])
gc.collect()


def _slice_sum(lo_year, hi_year):
    """Row sums over [lo_year, hi_year), clipped to the matrix range."""
    lo = YCOL.get(max(lo_year, YEAR_MIN))
    hi = YCOL.get(min(hi_year, YEAR_MAX + 1))
    if lo is None or hi is None or hi <= lo:
        return np.zeros(PUBS.shape[0], dtype=np.int32)
    return PUBS[:, lo:hi].sum(axis=1)


def pre_windows_at(year):
    """Total, recent and early counts, and the annualised growth between them.

    Growth compares the rate in the years immediately before the event with the
    rate in the years preceding those, so an author whose output is rising is
    matched to a candidate whose output is also rising.
    """
    recent = _slice_sum(year - RECENT_WINDOW, year)
    early = _slice_sum(year - PRE_WINDOW, year - RECENT_WINDOW)
    growth = (np.log1p(recent / RECENT_WINDOW)
              - np.log1p(early / EARLY_WINDOW))
    return recent + early, recent, early, growth

publication matrix: 8,118,192 candidates x 37 years (1.12 GiB)


## Matching

Treated authors are processed in random order under a fixed seed. In index
order the first authors take the closest available controls and the last take
what remains, which makes match quality correlate with author identifier.

Reuse is tracked per author rather than per row. The pool is expanded across
years, so a row-level counter would allow one person to serve as a control for
several treated authors in different years.

In [8]:
def match(treated, sc, ratio=MATCH_RATIO,
          with_replacement=WITH_REPLACEMENT, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)

    first_pub = sc.first_pub_year.to_numpy()
    cand_ids = sc.author_id.to_numpy()
    cand_income = sc.income_group.to_numpy()
    taken = np.zeros(len(sc), dtype=bool)

    years = sorted(treated.first_retraction_year.dropna().unique().astype(int))
    print(f"pseudo-retraction years  {years[0]}-{years[-1]}  ({len(years)})")

    use_income = ("income_group" in EXACT_ON
                  and not sc.income_group.isna().all()
                  and not treated.income_group.isna().all())
    use_band = "career_band" in EXACT_ON
    if not use_income:
        print("  [!] income_group unusable on one side; dropped from the key")

    matches, unmatched = [], 0

    for y in years:
        t_y = treated[treated.first_retraction_year == y]
        if t_y.empty:
            continue
        t_y = t_y.sample(frac=1.0, random_state=rng.integers(1 << 31))

        # Candidates whose observable career had begun by this year.
        career = y - first_pub
        eligible = (career >= 0) & (~taken)
        if not eligible.any():
            unmatched += len(t_y)
            continue

        pre, pre_recent, pre_early, pre_growth = pre_windows_at(y)
        band = pd.cut(career, bins=[0, 5, 10, 20, np.inf],
                      labels=["early (<5y)", "mid (5-10y)",
                              "senior (10-20y)", "veteran (20y+)"],
                      right=False).astype(object)

        idx_all = np.flatnonzero(eligible)
        keys = []
        for i in idx_all:
            k = []
            if use_band:
                k.append(band[i])
            if use_income:
                k.append(cand_income[i])
            keys.append(tuple(k))

        buckets = {}
        for pos, k in zip(idx_all, keys):
            buckets.setdefault(k, []).append(pos)
        buckets = {k: np.array(v) for k, v in buckets.items()}

        n_matched_y = 0
        for _, t in t_y.iterrows():
            k = []
            if use_band:
                k.append(str(t.career_band))
            if use_income:
                k.append(t.income_group)
            idx = buckets.get(tuple(k))
            if idx is None or not len(idx):
                unmatched += 1; continue

            if not with_replacement:
                idx = idx[~taken[idx]]
                if not len(idx):
                    unmatched += 1; continue

            # Level in each window, and the direction between them.
            d_recent = np.abs(np.log1p(pre_recent[idx] / RECENT_WINDOW)
                              - np.log1p(t.pre_recent / RECENT_WINDOW))
            d_early = np.abs(np.log1p(pre_early[idx] / EARLY_WINDOW)
                             - np.log1p(t.pre_early / EARLY_WINDOW))
            d_growth = np.abs(pre_growth[idx] - t.pre_growth)
            d_pub = np.abs(np.log1p(pre[idx]) - np.log1p(t.pre_publications))

            ok = ((d_recent <= CALIPER_RECENT) &
                  (d_early <= CALIPER_EARLY) &
                  (d_growth <= CALIPER_GROWTH))
            if pd.notna(t.get("career_years_at_retraction")):
                d_car = np.abs(career[idx] - t.career_years_at_retraction)
                ok = ok & (d_car <= CALIPER_CAREER)
            else:
                d_car = np.zeros(len(idx))

            if not ok.any():
                unmatched += 1; continue

            dist = (d_recent[ok] + d_early[ok] + d_growth[ok]
                    + 0.1 * d_car[ok])
            order = np.argsort(dist)[:ratio]
            chosen = idx[ok][order]

            for rank, cidx in enumerate(chosen):
                matches.append({
                    "treated_id": t.author_id,
                    "control_id": cand_ids[cidx],
                    "pseudo_retraction_year": y,
                    "arm": t.get("arm"),
                    "treated_position": t.get("first_position"),
                    "career_band": str(t.career_band),
                    "income_group": t.income_group,
                    "treated_pre_pubs": int(t.pre_publications),
                    "control_pre_pubs": int(pre[cidx]),
                    "treated_pre_recent": int(t.pre_recent),
                    "control_pre_recent": int(pre_recent[cidx]),
                    "treated_pre_early": int(t.pre_early),
                    "control_pre_early": int(pre_early[cidx]),
                    "treated_pre_growth": float(t.pre_growth),
                    "control_pre_growth": float(pre_growth[cidx]),
                    "match_distance": float(dist[order[rank]]),
                })
                if not with_replacement:
                    taken[cidx] = True
            n_matched_y += 1

        print(f"  {y}: {len(t_y):,} treated, {n_matched_y:,} matched "
              f"({n_matched_y / len(t_y):.1%}), "
              f"{int((~taken).sum()):,} candidates remaining")

    m = pd.DataFrame(matches)
    if m.empty:
        print("\n[!] no matches")
        return m

    print(f"\nmatched   {m.treated_id.nunique():,} of {len(treated):,} "
          f"({m.treated_id.nunique() / len(treated):.1%})")
    print(f"unmatched {unmatched:,}")
    print(f"controls  {m.control_id.nunique():,} unique")
    return m


t0 = time.time()
m = match(treated, sc)
print(f"elapsed {(time.time() - t0) / 60:.1f} min")

pseudo-retraction years  2015-2022  (8)
  2015: 3,346 treated, 3,344 matched (99.9%), 8,114,848 candidates remaining
  2016: 3,418 treated, 3,416 matched (99.9%), 8,111,432 candidates remaining
  2017: 3,544 treated, 3,544 matched (100.0%), 8,107,888 candidates remaining
  2018: 3,861 treated, 3,860 matched (100.0%), 8,104,028 candidates remaining
  2019: 5,101 treated, 5,098 matched (99.9%), 8,098,930 candidates remaining
  2020: 6,223 treated, 6,221 matched (100.0%), 8,092,709 candidates remaining
  2021: 9,584 treated, 9,582 matched (100.0%), 8,083,127 candidates remaining
  2022: 13,722 treated, 13,721 matched (100.0%), 8,069,406 candidates remaining

matched   48,786 of 48,799 (100.0%)
unmatched 13
controls  48,786 unique
elapsed 96.5 min


## Balance

In [9]:
if m.empty:
    print("no matches; nothing to assess")
else:
    a, b = m.treated_pre_pubs, m.control_pre_pubs
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    std_diff = (b.mean() - a.mean()) / pooled if pooled else 0.0

    print("pre-retraction publications")
    print(f"  treated mean            {a.mean():.2f}")
    print(f"  control mean            {b.mean():.2f}")
    print(f"  standardised difference {std_diff:+.4f}   "
          f"(conventional threshold 0.1)")

    for col_t, col_c, label in [
            ("treated_pre_recent", "control_pre_recent",
             f"publications, most recent {RECENT_WINDOW} years"),
            ("treated_pre_early", "control_pre_early",
             f"publications, earlier {EARLY_WINDOW} years"),
            ("treated_pre_growth", "control_pre_growth",
             "pre-period growth, log rate ratio")]:
        if col_t not in m.columns:
            continue
        x, z = m[col_t], m[col_c]
        pooled = np.sqrt((x.var(ddof=1) + z.var(ddof=1)) / 2)
        sd = (z.mean() - x.mean()) / pooled if pooled else 0.0
        print(f"\n{label}")
        print(f"  treated mean            {x.mean():.3f}")
        print(f"  control mean            {z.mean():.3f}")
        print(f"  standardised difference {sd:+.4f}")

    if "treated_pre_growth" in m.columns:
        share_t = float((m.treated_pre_growth > 0).mean())
        share_c = float((m.control_pre_growth > 0).mean())
        print(f"\nrising into the event")
        print(f"  treated {share_t:.1%}, controls {share_c:.1%}")

    print("\nmatch distance")
    print(f"  median  {m.match_distance.median():.4f}")
    print(f"  mean    {m.match_distance.mean():.4f}")
    print(f"  exact   {(m.match_distance == 0).mean():.1%} of pairs")

    for col, label in [("arm", "arm"), ("career_band", "career band"),
                       ("income_group", "income group")]:
        t_counts = (treated.arm if col == "arm" else treated[col]).value_counts()
        cov = pd.DataFrame({"treated": t_counts,
                            "matched": m[col].value_counts()}).fillna(0).astype(int)
        cov["rate"] = (100 * cov.matched / cov.treated).round(1)
        print(f"\ncoverage by {label}")
        print(cov.to_string())

    reuse = m.control_id.value_counts()
    n_reused = int((reuse > 1).sum())
    print(f"\ncontrols used more than once: {n_reused:,}"
          + ("  [!] unexpected without replacement"
             if n_reused and not WITH_REPLACEMENT else ""))

pre-retraction publications
  treated mean            40.64
  control mean            40.63
  standardised difference -0.0002   (conventional threshold 0.1)

publications, most recent 3 years
  treated mean            20.565
  control mean            20.549
  standardised difference -0.0005

publications, earlier 4 years
  treated mean            20.071
  control mean            20.076
  standardised difference +0.0001

pre-period growth, log rate ratio
  treated mean            0.286
  control mean            0.286
  standardised difference -0.0002

rising into the event
  treated 68.9%, controls 68.9%

match distance
  median  0.0000
  mean    0.0085
  exact   89.1% of pairs

coverage by arm
                      treated  matched   rate
arm                                          
AUTHOR_MISCONDUCT       25368    25365  100.0
HONEST_ERROR            10928    10923  100.0
EDITORIAL_COMPROMISE     7132     7128   99.9
UNCONFIRMED_CONCERNS     4303     4302  100.0
ETHICS_VIOLATION     

## Write

In [10]:
if m.empty:
    print("no matches; nothing to write")
else:
    sc.drop(columns=["counts_by_year"], errors="ignore").to_csv(
        OUT_SCREENED, index=False)
    print(f"{OUT_SCREENED}: {len(sc):,} rows")

    m.to_csv(OUT_MATCHES, index=False)
    print(f"{OUT_MATCHES}: {len(m):,} pairs")

    meta = sc.set_index("author_id")
    q = (m[["control_id", "pseudo_retraction_year", "arm", "career_band",
            "income_group"]]
         .drop_duplicates(subset="control_id")
         .rename(columns={"control_id": "author_id",
                          "pseudo_retraction_year": "first_retraction_year",
                          "arm": "matched_arm"}))
    q["first_category"] = "CONTROL"
    q["first_position"] = None
    q["n_retractions"] = 0
    for col in ["source_id", "last_country", "works_count", "first_pub_year"]:
        if col in meta.columns:
            q[col] = q.author_id.map(meta[col])

    q.to_csv(OUT_QUEUE, index=False)
    print(f"{OUT_QUEUE}: {len(q):,} controls")

    print(f"\ncontrols by the arm they were matched to")
    print(q.matched_arm.value_counts().to_string())
    print(f"\nby pseudo-retraction year")
    print(q.first_retraction_year.value_counts().sort_index().to_string())

data/interim/phase06_candidates_screened.csv: 8,118,192 rows
data/interim/phase06_matches.csv: 48,786 pairs
data/interim/phase07_control_queue.csv: 48,786 controls

controls by the arm they were matched to
matched_arm
AUTHOR_MISCONDUCT       25365
HONEST_ERROR            10923
EDITORIAL_COMPROMISE     7128
UNCONFIRMED_CONCERNS     4302
ETHICS_VIOLATION          723
UNCLASSIFIED              345

by pseudo-retraction year
first_retraction_year
2015     3344
2016     3416
2017     3544
2018     3860
2019     5098
2020     6221
2021     9582
2022    13721
